# 03 Batter Weakness Profiles

Build batter pitch-type weakness profiles.



In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import save_csv

print(PROJECT_ROOT)

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting


In [2]:
pa_path = PROJECT_ROOT / "data/processed/pa_2025_04_01_to_05_31.csv"

pa_df = pd.read_csv(pa_path)
pa_df["game_date"] = pd.to_datetime(pa_df["game_date"])

print(pa_df.shape)
pa_df.head()

(60470, 13)


,game_date,batter,pitcher,events,description,pitch_type,stand,p_throws,launch_speed,launch_angle,estimated_woba_using_speedangle,woba_value,xwoba_value
0,2025-05-31,691785,606160,field_out,hit_into_play,FF,L,R,91.3,33.0,0.057000,0.0,0.057000
1,2025-05-31,665966,606160,field_out,hit_into_play,SL,R,R,71.4,-20.0,0.036000,0.0,0.036000
2,2025-05-31,677800,606160,field_out,hit_into_play,FF,L,R,103.4,36.0,0.975773,0.0,0.975773
3,2025-05-31,663897,595897,strikeout,swinging_strike,SL,R,R,NaN,NaN,0.000000,0.0,0.000000
4,2025-05-31,671739,595897,field_out,hit_into_play,SL,L,R,100.8,4.0,0.542000,0.0,0.542000


In [3]:
needed_cols = [
    "game_date",
    "batter",
    "pitcher",
    "events",
    "pitch_type",
    "xwoba_value",
    "woba_value",
]

missing_cols = [c for c in needed_cols if c not in pa_df.columns]
print("Missing columns:", missing_cols)

Missing columns: []


In [4]:
pa_df["pitch_type"].value_counts(dropna=False)

pitch_type
FF     18453
SI     10087
SL      8806
CH      6508
ST      4571
FC      4484
CU      3286
FS      2380
KC      1031
SV       301
NaN      194
FO       134
EP       110
FA        87
CS        33
SC         3
PO         1
KN         1
Name: count, dtype: int64

In [5]:
def map_pitch_group(pitch_type):
    if pd.isna(pitch_type):
        return "unknown"

    fastball = {"FF", "FA"}
    sinker = {"SI"}
    cutter = {"FC"}
    slider = {"SL", "ST"}
    curveball = {"CU", "KC", "SV"}
    changeup = {"CH", "FS", "FO"}

    if pitch_type in fastball:
        return "fastball"
    if pitch_type in sinker:
        return "sinker"
    if pitch_type in cutter:
        return "cutter"
    if pitch_type in slider:
        return "slider"
    if pitch_type in curveball:
        return "curveball"
    if pitch_type in changeup:
        return "changeup"

    return "other"


pa_df["pitch_group"] = pa_df["pitch_type"].apply(map_pitch_group)

pa_df["pitch_group"].value_counts()

pitch_group
fastball     18540
slider       13377
sinker       10087
changeup      9022
curveball     4618
cutter        4484
unknown        194
other          148
Name: count, dtype: int64

In [6]:
league_pitch_group = (
    pa_df
    .groupby("pitch_group")
    .agg(
        league_PA=("events", "count"),
        league_xwOBA=("xwoba_value", "mean"),
        league_wOBA=("woba_value", "mean"),
    )
    .reset_index()
    .sort_values("league_PA", ascending=False)
)

league_pitch_group

,pitch_group,league_PA,league_xwOBA,league_wOBA
3,fastball,18540,0.348667,0.342066
6,slider,13377,0.286038,0.289243
5,sinker,10087,0.344215,0.354377
0,changeup,9022,0.275975,0.278863
1,curveball,4618,0.273359,0.292822
2,cutter,4484,0.344330,0.342808
7,unknown,194,0.392521,0.392784
4,other,148,0.366375,0.448311


In [7]:
batter_pitch_group = (
    pa_df
    .groupby(["batter", "pitch_group"])
    .agg(
        PA=("events", "count"),
        xwOBA=("xwoba_value", "mean"),
        wOBA=("woba_value", "mean"),
    )
    .reset_index()
)

batter_pitch_group.head()

,batter,pitch_group,PA,xwOBA,wOBA
0,455117,changeup,6,0.295794,0.208333
1,455117,curveball,2,0.004500,0.000000
2,455117,cutter,4,0.064750,0.225000
3,455117,fastball,28,0.339023,0.250000
4,455117,sinker,16,0.273320,0.225000


In [8]:
batter_pitch_group = batter_pitch_group.merge(
    league_pitch_group[["pitch_group", "league_xwOBA", "league_wOBA"]],
    on="pitch_group",
    how="left"
)

batter_pitch_group["xwOBA_vs_league"] = (
    batter_pitch_group["xwOBA"] - batter_pitch_group["league_xwOBA"]
)

batter_pitch_group["weakness_score"] = (
    batter_pitch_group["league_xwOBA"] - batter_pitch_group["xwOBA"]
)

batter_pitch_group.head()

,batter,pitch_group,PA,xwOBA,wOBA,league_xwOBA,league_wOBA,xwOBA_vs_league,weakness_score
0,455117,changeup,6,0.295794,0.208333,0.275975,0.278863,0.019819,-0.019819
1,455117,curveball,2,0.004500,0.000000,0.273359,0.292822,-0.268859,0.268859
2,455117,cutter,4,0.064750,0.225000,0.344330,0.342808,-0.279580,0.279580
3,455117,fastball,28,0.339023,0.250000,0.348667,0.342066,-0.009644,0.009644
4,455117,sinker,16,0.273320,0.225000,0.344215,0.354377,-0.070895,0.070895


In [9]:
min_pa_by_pitch_group = 5

batter_pitch_group["reliable_sample"] = (
    batter_pitch_group["PA"] >= min_pa_by_pitch_group
)

batter_pitch_group.sort_values("weakness_score", ascending=False).head(20)

,batter,pitch_group,PA,xwOBA,wOBA,league_xwOBA,league_wOBA,xwOBA_vs_league,weakness_score,reliable_sample
2893,687597,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
1135,660162,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
1491,665750,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
111,527038,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
449,606192,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
2459,680574,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
1672,667472,unknown,1,0.000,0.0,0.392521,0.392784,-0.392521,0.392521,False
1011,656180,other,1,0.000,0.0,0.366375,0.448311,-0.366375,0.366375,False
667,624428,other,1,0.000,0.0,0.366375,0.448311,-0.366375,0.366375,False
2035,670770,other,1,0.000,0.0,0.366375,0.448311,-0.366375,0.366375,False


In [10]:
xwoba_wide = (
    batter_pitch_group
    .pivot(index="batter", columns="pitch_group", values="xwOBA")
    .add_prefix("xwOBA_vs_")
    .reset_index()
)

weakness_wide = (
    batter_pitch_group
    .pivot(index="batter", columns="pitch_group", values="weakness_score")
    .add_prefix("weakness_vs_")
    .reset_index()
)

pa_wide = (
    batter_pitch_group
    .pivot(index="batter", columns="pitch_group", values="PA")
    .add_prefix("PA_vs_")
    .reset_index()
)

In [11]:
batter_weakness_profile = (
    xwoba_wide
    .merge(weakness_wide, on="batter", how="outer")
    .merge(pa_wide, on="batter", how="outer")
)

print(batter_weakness_profile.shape)
batter_weakness_profile.head()

(535, 25)


pitch_group,batter,xwOBA_vs_changeup,xwOBA_vs_curveball,xwOBA_vs_cutter,xwOBA_vs_fastball,xwOBA_vs_other,xwOBA_vs_sinker,xwOBA_vs_slider,xwOBA_vs_unknown,weakness_vs_changeup,...,weakness_vs_slider,weakness_vs_unknown,PA_vs_changeup,PA_vs_curveball,PA_vs_cutter,PA_vs_fastball,PA_vs_other,PA_vs_sinker,PA_vs_slider,PA_vs_unknown
0,455117,0.295794,0.004500,0.064750,0.339023,NaN,0.273320,0.225133,NaN,-0.019819,...,0.060904,NaN,6.0,2.0,4.0,28.0,NaN,16.0,15.0,NaN
1,456781,0.193750,0.311058,0.360000,0.269775,NaN,0.349444,0.158916,NaN,0.082225,...,0.127122,NaN,14.0,8.0,5.0,24.0,NaN,9.0,13.0,NaN
2,457705,0.364941,0.272429,0.479428,0.324979,NaN,0.455160,0.245250,0.4,-0.088966,...,0.040788,-0.007479,26.0,7.0,14.0,59.0,NaN,39.0,42.0,1.0
3,457759,0.220200,0.304000,0.467659,0.296534,NaN,0.396699,0.299956,NaN,0.055775,...,-0.013919,NaN,10.0,2.0,6.0,35.0,NaN,14.0,24.0,NaN
4,467793,0.249928,0.224316,0.310714,0.305895,0.46,0.404974,0.322223,NaN,0.026047,...,-0.036185,NaN,34.0,12.0,14.0,80.0,1.0,32.0,31.0,NaN


In [12]:
reliable = batter_pitch_group[batter_pitch_group["reliable_sample"]].copy()

main_weakness = (
    reliable
    .sort_values(["batter", "weakness_score"], ascending=[True, False])
    .groupby("batter")
    .head(1)
    .rename(columns={
        "pitch_group": "main_weakness_pitch_group",
        "weakness_score": "main_weakness_score",
        "xwOBA": "main_weakness_xwOBA",
        "league_xwOBA": "main_weakness_league_xwOBA",
        "PA": "main_weakness_PA",
    })
    [[
        "batter",
        "main_weakness_pitch_group",
        "main_weakness_score",
        "main_weakness_xwOBA",
        "main_weakness_league_xwOBA",
        "main_weakness_PA",
    ]]
)

main_weakness.head()

,batter,main_weakness_pitch_group,main_weakness_score,main_weakness_xwOBA,main_weakness_league_xwOBA,main_weakness_PA
4,455117,sinker,0.070895,0.273320,0.344215,16
11,456781,slider,0.127122,0.158916,0.286038,13
17,457705,slider,0.040788,0.245250,0.286038,42
19,457759,changeup,0.055775,0.220200,0.275975,10
26,467793,curveball,0.049043,0.224316,0.273359,12


In [13]:
batter_weakness_profile = batter_weakness_profile.merge(
    main_weakness,
    on="batter",
    how="left"
)

batter_weakness_profile.head()

,batter,xwOBA_vs_changeup,xwOBA_vs_curveball,xwOBA_vs_cutter,xwOBA_vs_fastball,xwOBA_vs_other,xwOBA_vs_sinker,xwOBA_vs_slider,xwOBA_vs_unknown,weakness_vs_changeup,...,PA_vs_fastball,PA_vs_other,PA_vs_sinker,PA_vs_slider,PA_vs_unknown,main_weakness_pitch_group,main_weakness_score,main_weakness_xwOBA,main_weakness_league_xwOBA,main_weakness_PA
0,455117,0.295794,0.004500,0.064750,0.339023,NaN,0.273320,0.225133,NaN,-0.019819,...,28.0,NaN,16.0,15.0,NaN,sinker,0.070895,0.273320,0.344215,16.0
1,456781,0.193750,0.311058,0.360000,0.269775,NaN,0.349444,0.158916,NaN,0.082225,...,24.0,NaN,9.0,13.0,NaN,slider,0.127122,0.158916,0.286038,13.0
2,457705,0.364941,0.272429,0.479428,0.324979,NaN,0.455160,0.245250,0.4,-0.088966,...,59.0,NaN,39.0,42.0,1.0,slider,0.040788,0.245250,0.286038,42.0
3,457759,0.220200,0.304000,0.467659,0.296534,NaN,0.396699,0.299956,NaN,0.055775,...,35.0,NaN,14.0,24.0,NaN,changeup,0.055775,0.220200,0.275975,10.0
4,467793,0.249928,0.224316,0.310714,0.305895,0.46,0.404974,0.322223,NaN,0.026047,...,80.0,1.0,32.0,31.0,NaN,curveball,0.049043,0.224316,0.273359,12.0


In [14]:
profile_path = PROJECT_ROOT / "data/features/batter_weakness_profiles_2025_04_01_to_05_31.csv"
save_csv(batter_weakness_profile, profile_path)

long_path = PROJECT_ROOT / "data/features/batter_pitch_group_long_2025_04_01_to_05_31.csv"
save_csv(batter_pitch_group, long_path)

league_path = PROJECT_ROOT / "data/features/league_pitch_group_2025_04_01_to_05_31.csv"
save_csv(league_pitch_group, league_path)

print(profile_path.exists())
print(long_path.exists())
print(league_path.exists())

True
True
True


## Summary

This notebook builds batter pitch-type weakness profiles from PA-level Statcast data.

For each hitter and pitch group, we calculate:
- PA
- xwOBA
- wOBA
- league-average xwOBA for that pitch group
- xwOBA difference relative to league average
- weakness score = league xwOBA - hitter xwOBA

A positive weakness score indicates that the hitter performs worse than league average against that pitch group.

The final wide-format profile will be used in the matchup adjustment module.